# MIT-BIH RR intervals to BPM categories

This Colab-ready notebook downloads beat annotations from the MIT-BIH Arrhythmia Database through the official `wfdb` package, converts consecutive beat intervals to BPM, and creates sample-level and 10-value window datasets.

> **Scope:** these labels describe simplified heart-rate ranges. BPM-only features do not retain ECG morphology and cannot reproduce clinical arrhythmia diagnosis. Keep record IDs in the data so evaluation can be split by record rather than randomly leaking the same record across train and test sets.

Sources: [MIT-BIH on PhysioNet](https://physionet.org/content/mitdb/1.0.0/) and the [WFDB Python documentation](https://wfdb.readthedocs.io/en/latest/).

In [ ]:
%pip -q install wfdb pandas numpy matplotlib

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Start with five records for a quick run. Replace with MIT_BIH_RECORDS
# below when you are ready to process all 48 records.
RECORDS = ['100', '101', '102', '103', '104']
MIT_BIH_RECORDS = [
    '100', '101', '102', '103', '104', '105', '106', '107',
    '108', '109', '111', '112', '113', '114', '115', '116',
    '117', '118', '119', '121', '122', '123', '124', '200',
    '201', '202', '203', '205', '207', '208', '209', '210',
    '212', '213', '214', '215', '217', '219', '220', '221',
    '222', '223', '228', '230', '231', '232', '233', '234',
]

# Symbols used by the MIT-BIH annotation files for beat locations.
# Non-beat rhythm-change and noise annotations are intentionally excluded.
BEAT_SYMBOLS = {
    'N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F',
    'e', 'j', 'n', 'E', '/', 'f', 'Q', '?',
}
MIN_BPM = 30.0
MAX_BPM = 220.0
WINDOW_SIZE = 10
WINDOW_STRIDE = 5

In [ ]:
def rate_category(bpm: float) -> str:
    if bpm < 60.0:
        return 'Bradycardia'
    if bpm <= 100.0:
        return 'Normal'
    return 'Tachycardia'


def read_record_bpm(record_name: str) -> pd.DataFrame:
    annotation = wfdb.rdann(record_name, 'atr', pn_dir='mitdb')
    beat_samples = np.asarray([
        sample
        for sample, symbol in zip(annotation.sample, annotation.symbol)
        if symbol in BEAT_SYMBOLS
    ], dtype=np.int64)

    beat_times_s = beat_samples / float(annotation.fs)
    rr_interval_s = np.diff(beat_times_s)
    bpm = 60.0 / rr_interval_s
    valid = np.isfinite(bpm) & (bpm >= MIN_BPM) & (bpm <= MAX_BPM)

    frame = pd.DataFrame({
        'record': record_name,
        'time_s': beat_times_s[1:][valid],
        'rr_interval_s': rr_interval_s[valid],
        'bpm': bpm[valid],
    })
    frame['rate_class'] = frame['bpm'].map(rate_category)
    return frame


sample_frames = [read_record_bpm(record) for record in RECORDS]
samples = pd.concat(sample_frames, ignore_index=True)
samples.to_csv(OUTPUT_DIR / 'mit_bih_bpm_samples.csv', index=False)

print(f'Created {len(samples):,} valid RR-derived BPM samples.')
display(samples.head())
display(samples.groupby(['record', 'rate_class']).size().unstack(fill_value=0))

In [ ]:
def create_windows(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    feature_names = [f'bpm_{i}' for i in range(WINDOW_SIZE)]

    # Construct every window inside one record. Do not cross record boundaries.
    for record, group in frame.groupby('record', sort=True):
        ordered = group.sort_values('time_s').reset_index(drop=True)
        values = ordered['bpm'].to_numpy(dtype=np.float32)
        times = ordered['time_s'].to_numpy(dtype=np.float64)

        for start in range(0, len(values) - WINDOW_SIZE + 1, WINDOW_STRIDE):
            window = values[start:start + WINDOW_SIZE]
            median_bpm = float(np.median(window))
            row = {
                'record': record,
                'start_time_s': float(times[start]),
                'label': rate_category(median_bpm),
            }
            row.update(dict(zip(feature_names, window.tolist())))
            rows.append(row)

    return pd.DataFrame(rows)


windows = create_windows(samples)
windows.to_csv(OUTPUT_DIR / 'mit_bih_bpm_windows.csv', index=False)
print(f'Created {len(windows):,} windows of {WINDOW_SIZE} BPM values.')
display(windows.head())
display(windows['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
samples['bpm'].plot.hist(bins=60, ax=axes[0], color='#2563eb')
axes[0].set(title='RR-derived BPM distribution', xlabel='BPM')
windows['label'].value_counts().reindex(
    ['Bradycardia', 'Normal', 'Tachycardia'], fill_value=0
).plot.bar(ax=axes[1], color=['#f59e0b', '#10b981', '#ef4444'])
axes[1].set(title='Window labels', xlabel='Rate category', ylabel='Windows')
plt.tight_layout()
plt.show()

## Before training

Use `record` as the grouping key for train/validation/test splits. A random row split can place overlapping windows or the same patient's record on both sides of evaluation and inflate the score. Confirm that the Edge Impulse input order is `bpm_0` through `bpm_9`, exactly matching the firmware window.

The exported model package is not generated by this notebook. Train in Edge Impulse, export the complete Arduino/C++ library, and install it under `firmware/lib/arrhythmia_inferencing/`.